# OpenAI Formatter
> Format a ContextWindow into the OpenAI Chat Completions structure.

`OpenAIFormatter` converts your context into the `messages` list
expected by the OpenAI Python SDK.

**Time:** ~5 minutes

## Setup

In [ ]:
from anchor.formatters import OpenAIFormatter
from anchor.models import ContextWindow, ContextItem, SourceType

## Build a Sample Context Window

In [ ]:
from anchor.models import ContextWindow, ContextItem, SourceType

# Build a sample context window with mixed sources
window = ContextWindow(max_tokens=4096)
window.add_item(ContextItem(
    id="sys-1",
    content="You are helpful.",
    source=SourceType.SYSTEM,
    score=1.0,
    priority=10,
    token_count=5,
))
window.add_item(ContextItem(
    id="doc-1",
    content="Python is a language.",
    source=SourceType.RETRIEVAL,
    score=0.9,
    priority=5,
    token_count=6,
))

print(f"Context window: {len(window.items)} items, "
      f"budget {window.max_tokens} tokens")
for item in window.items:
    print(f"  [{item.source.value}] {item.id}: {item.content!r}")

## Format for the OpenAI API
The formatter returns a dict containing a `messages` list with
role/content pairs.

In [ ]:
formatter = OpenAIFormatter()
print(f"Format type: {formatter.format_type}")

In [ ]:
output = formatter.format(window)
print(type(output))
print(output)

## Walk Through the Messages
Each context item is mapped to an appropriate chat role.

In [ ]:
if isinstance(output, dict) and "messages" in output:
    for i, msg in enumerate(output["messages"]):
        print(f"Message {i}: role={msg.get('role')}, "
              f"content={msg.get('content')!r}")
else:
    print(output)

## Compare with Anthropic
Anchor's formatters share the same `Formatter` protocol, so
switching providers is a one-line change.

In [ ]:
from anchor.formatters import AnthropicFormatter

anthropic_out = AnthropicFormatter().format(window)
openai_out = OpenAIFormatter().format(window)

print("Anthropic output type:", type(anthropic_out).__name__)
print("OpenAI output type:   ", type(openai_out).__name__)
print()
print("Same Formatter protocol, different providers.")

## Key Takeaways

- `OpenAIFormatter` produces a `messages` list compatible with OpenAI Chat Completions.
- System items map to the `system` role; retrieval items map to `user`.
- Swapping formatters requires no changes to your context pipeline.